In [1]:
!ls /kaggle/input

datasets


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image

In [3]:
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Workspace initialized. Using device: {device}")

Workspace initialized. Using device: cuda


In [5]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
# Forces cuDNN to use deterministic algorithms
torch.backends.cudnn.deterministic = True 
torch.backends.cudnn.benchmark = False

In [ ]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torchvision.transforms as transforms

class TartanAirDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        
        # TartanAir usually structures data with 'image_left' and 'depth_left' folders
        # We use recursive globbing to find all files and sort them to guarantee perfect alignment
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, '**/image_left/*.png'), recursive=True))
        
        # Depth maps in TartanAir are often stored as exact float values in .npy files
        self.depth_paths = sorted(glob.glob(os.path.join(root_dir, '**/depth_left/*.npy'), recursive=True))
        
        # Fallback just in case this specific Kaggle uploader saved depth as PNGs
        if len(self.depth_paths) == 0:
             self.depth_paths = sorted(glob.glob(os.path.join(root_dir, '**/depth_left/*.png'), recursive=True))

        print(f"Pipeline Linked: Found {len(self.image_paths)} RGB images and {len(self.depth_paths)} Depth maps.")
        
        # Sanity Check!
        assert len(self.image_paths) == len(self.depth_paths), "Mismatch between number of images and depth maps!"

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 1. Load RGB
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        # 2. Load Depth
        depth_path = self.depth_paths[idx]
        if depth_path.endswith('.npy'):
            depth = np.load(depth_path)
        else:
            depth = np.array(Image.open(depth_path), dtype=np.float32)
            
        # Convert depth to a PyTorch tensor with a channel dimension -> Shape: (1, H, W)
        depth_tensor = torch.from_numpy(depth).unsqueeze(0)

        # 3. Apply FastDepth specific transforms to the RGB image
        if self.transform:
            image = self.transform(image)
            # We must also resize the depth map to 224x224 to match FastDepth's output, 
            # but we DO NOT apply ImageNet color normalization to a depth map!
            depth_tensor = F.interpolate(depth_tensor.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False).squeeze(0)

        return image, depth_tensor

# --- PIPELINE INITIALIZATION ---

# Define the exact transform FastDepth requires
fastdepth_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Point the Dataset to the Kaggle folder you mounted
GASCOLA_PATH = "/kaggle/input/datasets/naz182/tartanair-gascola"

# Create the Dataset and DataLoader
val_dataset = TartanAirDataset(root_dir=GASCOLA_PATH, transform=fastdepth_transform)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

# --- VISUAL VERIFICATION ---

# Grab exactly one batch to test
sample_images, sample_depths = next(iter(val_loader))

# Un-normalize the image so pyplot can render it normally without looking deep-fried
def unnormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

plt.figure(figsize=(10, 5))

# Plot the corresponding Depth Map
plt.subplot(1, 2, 2)
plt.title("Ground Truth Depth (Clipped to 50m)")

# Extract the raw depth array
depth_array = sample_depths[0].squeeze(0).numpy()

# Clip the massive sky values down to 50 so they don't ruin our color scale
depth_clipped = np.clip(depth_array, a_min=0, a_max=50)

plt.imshow(depth_clipped, cmap='magma')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torchvision.models as models

print("--- Procuring Baseline Architecture ---")
# 1. Load a pre-trained continuous backbone (MobileNetV2 is standard for lightweight AV tasks)
# We are using this as a proxy for the FastDepth encoder
continuous_model = models.mobilenet_v2(weights='DEFAULT')

# 2. The SNN-Ready Refactoring Script
def make_snn_ready(module):
    """
    Recursively hunts down activation functions and ensures they are physically
    instantiated and have inplace=False so our SNN hooks can read their voltages.
    """
    for child_name, child in module.named_children():
        # FastDepth/MobileNet often use ReLU6, which we convert to standard ReLU for SNNs
        if isinstance(child, nn.ReLU) or isinstance(child, nn.ReLU6):
            # Physically overwrite the layer
            setattr(module, child_name, nn.ReLU(inplace=False))
        else:
            # Recursively dive deeper into the sub-blocks
            make_snn_ready(child)

# 3. Execute the Neuromorphic Surgery Prep
print("Refactoring architecture to SNN-Ready state...")
make_snn_ready(continuous_model)

# 4. Lock the model (CRITICAL)
# This freezes Batch Norm and Dropout so our baseline is deterministic
continuous_model.eval()

# Move the model to your Kaggle GPU
continuous_model = continuous_model.to(device)

print("Model locked, loaded, and successfully mounted to GPU.")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm

# 1. Build the Depth Decoder
class SimpleDepthDecoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        # We extract just the feature layers (removes the 1000-class classification head)
        self.encoder = backbone.features
        
        # A simple upsampling block to scale the 7x7 feature map back to 224x224
        self.decoder = nn.Sequential(
            nn.Conv2d(1280, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=False),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), # 14x14
            
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=False),
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False), # 56x56
            
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=False),
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False), # 224x224
            
            nn.Conv2d(64, 1, kernel_size=3, padding=1),
            nn.ReLU(inplace=False) # Depth is always positive, so we cap with a ReLU
        )

    def forward(self, x):
        features = self.encoder(x)
        depth_map = self.decoder(features)
        return depth_map

# Assemble the full continuous depth model and lock it
depth_model = SimpleDepthDecoder(continuous_model).to(device)
depth_model.eval()

# 2. Define the Evaluation Metric
def calculate_rmse(pred, target):
    mse = F.mse_loss(pred, target)
    return torch.sqrt(mse).item()

# 3. The Validation Loop
print("--- Starting Phase 1 Baseline Inference ---")
total_rmse = 0.0
num_batches = 0
MAX_BATCHES = 100 # Stop after 1,600 images to save time

with torch.no_grad():
    for images, gt_depths in tqdm(val_loader, total=MAX_BATCHES, desc="Calculating Golden RMSE"):
        # CRITICAL: Send data to the Kaggle GPU
        images = images.to(device)
        gt_depths = gt_depths.to(device)
        
        # Forward pass through the heavy continuous math
        predicted_depths = depth_model(images)
        
        # Calculate how far off the continuous model is from reality
        batch_rmse = calculate_rmse(predicted_depths, gt_depths)
        total_rmse += batch_rmse
        num_batches += 1
        
        if num_batches >= MAX_BATCHES:
            break

golden_rmse = total_rmse / num_batches

print("\n=========================================")
print("          PHASE 1 COMPLETE               ")
print("=========================================")
print(f"Continuous Golden Baseline RMSE: {golden_rmse:.4f}")
print("=========================================")

In [ ]:
import numpy as np

# 1. Dictionary to store the raw voltage data
activation_profiles = {}

# 2. The Neuromorphic Probe (Forward Hook)
def get_activation_profile(layer_name):
    def hook(model, input, output):
        # The 'output' is the massive tensor of continuous activations (voltages)
        # We detach it from the GPU and flatten it into a 1D array
        flat_voltages = output.detach().cpu().numpy().flatten()
        
        # Storing millions of float32 pixels per layer will instantly crash Kaggle's RAM.
        # We randomly subsample 5% of the pixels—statistically perfect for finding percentiles.
        subsample = np.random.choice(flat_voltages, size=int(len(flat_voltages) * 0.05), replace=False)
        
        if layer_name not in activation_profiles:
            activation_profiles[layer_name] = []
        activation_profiles[layer_name].extend(subsample)
        
    return hook

# 3. Attach the Probes
hooks = []
print("--- Initiating Neuromorphic Profiling ---")

# We loop through the backbone we prepped in Phase 1
for name, module in continuous_model.named_modules():
    if isinstance(module, nn.ReLU):
        # Attach the hook and save the 'handle' so we can rip it out later
        handle = module.register_forward_hook(get_activation_profile(name))
        hooks.append(handle)
        print(f"Probe attached to: {name}")

print(f"\nSuccessfully attached {len(hooks)} probes. The network is wired.")

In [ ]:
from tqdm.notebook import tqdm

print("--- Running Calibration Pass ---")
CALIBRATION_BATCHES = 20 # ~320 images is plenty to get a stable distribution

with torch.no_grad():
    for i, (images, _) in enumerate(tqdm(val_loader, total=CALIBRATION_BATCHES, desc="Profiling Voltages")):
        images = images.to(device)
        
        # We only pass it through the continuous backbone, not the full depth decoder
        # The hooks intercept the data automatically in the background!
        _ = continuous_model(images)
        
        if i >= CALIBRATION_BATCHES - 1:
            break

print("Calibration complete. Profiling data secured.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Calculate the 99.9th percentile threshold for every layer
snn_thresholds = {}
print("--- Calculating SNN Scale-and-Fire Thresholds (99.9th Percentile) ---")

for layer_name, voltages in activation_profiles.items():
    # Convert list to numpy array for fast statistical operations
    voltage_array = np.array(voltages)
    
    # Calculate the 99.9th percentile (ignoring absolute outliers)
    threshold = np.percentile(voltage_array, 99.9)
    snn_thresholds[layer_name] = threshold
    
    print(f"Layer: {layer_name:25} | Optimal Threshold (\u03b8): {threshold:.4f}")

# 2. Visualize the Voltage Distribution of the First Layer
print("\n--- Plotting Activation Profile for First Layer ---")
first_layer = list(activation_profiles.keys())[0]
first_layer_voltages = np.array(activation_profiles[first_layer])

plt.figure(figsize=(10, 5))
plt.hist(first_layer_voltages, bins=100, color='darkviolet', alpha=0.7, edgecolor='black')
plt.axvline(snn_thresholds[first_layer], color='red', linestyle='--', linewidth=2, 
            label=f"99.9th Percentile (\u03b8 = {snn_thresholds[first_layer]:.4f})")

plt.title(f"Activation (Voltage) Profile: {first_layer}", fontsize=14)
plt.xlabel("Continuous Activation Value (V)", fontsize=12)
plt.ylabel("Frequency (Pixel Count)", fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.legend(fontsize=12)
plt.yscale('log') # Log scale because neural activations are highly skewed towards zero
plt.show()

In [ ]:
# 1. Remove the hooks so they don't intercept data during our final inference
for handle in hooks:
    handle.remove()
print("Probes successfully detached.")

# 2. Define the Single-Timestep Scale-and-Fire Neuron (SFN)
class ScaleAndFireNeuron(nn.Module):
    def __init__(self, threshold):
        super().__init__()
        self.threshold = threshold
        
    def forward(self, x):
        # The SNN Quantization Step:
        # If the continuous voltage (x) exceeds the threshold, fire a spike (1.0).
        # If it is below the threshold, stay silent (0.0).
        spikes = (x >= self.threshold).float()
        
        # In T=1 converted SNNs, the outgoing signal is the discrete spike scaled by the threshold.
        # This allows the subsequent layers to process it similarly to continuous activations.
        return spikes * self.threshold

# 3. The Neuromorphic Surgery: Swap ReLUs for SFNs
def convert_to_snn(module, prefix=""):
    """
    Recursively navigates the architecture. When it finds a ReLU, it replaces it 
    with an SFN initialized with the exact optimal threshold we calculated for that layer.
    """
    for name, child in module.named_children():
        # Reconstruct the exact layer name string (e.g., 'features.0.2')
        full_name = f"{prefix}.{name}" if prefix else name
        
        if isinstance(child, nn.ReLU):
            # Check if we have a profiled threshold for this layer
            if full_name in snn_thresholds:
                theta = snn_thresholds[full_name]
                # Overwrite the continuous layer with the Spiking layer
                setattr(module, name, ScaleAndFireNeuron(threshold=theta))
                print(f"Converted {full_name:25} -> Spiking SFN (\u03b8 = {theta:.4f})")
        else:
            # Dive deeper into nested blocks
            convert_to_snn(child, full_name)

print("\n--- Initiating Phase 3: SNN Conversion ---")
convert_to_snn(continuous_model)

# 4. Evaluate the new Spiking Neural Network
print("\n--- Running Final Phase 3 Inference ---")
total_snn_rmse = 0.0
num_batches = 0

# We use the same MAX_BATCHES limit as Phase 1 for a perfect 1:1 comparison
with torch.no_grad():
    for images, gt_depths in tqdm(val_loader, total=MAX_BATCHES, desc="Calculating SNN RMSE"):
        images = images.to(device)
        gt_depths = gt_depths.to(device)
        
        # Forward pass through the NOW SPIKING backbone + continuous decoder
        predicted_depths = depth_model(images)
        
        batch_rmse = calculate_rmse(predicted_depths, gt_depths)
        total_snn_rmse += batch_rmse
        num_batches += 1
        
        if num_batches >= MAX_BATCHES:
            break

snn_rmse = total_snn_rmse / num_batches

print("\n=========================================")
print("          PHASE 3 COMPLETE               ")
print("=========================================")
print(f"Continuous Golden Baseline: {golden_rmse:.4f}")
print(f"Single-Timestep SNN RMSE:   {snn_rmse:.4f}")
print("=========================================")

In [ ]:
import torch.nn.functional as F
import torchvision.models as models
import matplotlib.pyplot as plt

print("--- Initiating Feature-Level MSE Analysis ---")

# 1. Procure a fresh Continuous Encoder to compare against
# (Since we permanently mutated continuous_model into an SNN)
golden_encoder = models.mobilenet_v2(weights='DEFAULT').features.to(device)
golden_encoder.eval()

# Your mutated model is our Spiking Encoder
spiking_encoder = continuous_model.features

# 2. Grab a single batch of images from TartanAir
sample_images, _ = next(iter(val_loader))
sample_images = sample_images.to(device)

# 3. Extract the deep feature maps (Shape: [Batch, 1280 channels, 7 height, 7 width])
with torch.no_grad():
    continuous_features = golden_encoder(sample_images)
    spiking_features = spiking_encoder(sample_images)

# 4. Calculate the Feature-Level Mean Squared Error
feature_mse = F.mse_loss(spiking_features, continuous_features).item()

print(f"Feature-Level MSE (Continuous vs. SNN): {feature_mse:.6f}")

# --- Visual Proof ---
print("\n--- Generating Neuromorphic Feature Maps (Total Energy) ---")
# Average the activations across all 1280 channels to see the total extracted geometry
cont_map = continuous_features[0].mean(dim=0).cpu().numpy()
snn_map = spiking_features[0].mean(dim=0).cpu().numpy()

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title("Continuous Feature Map (Float32)")
plt.imshow(cont_map, cmap='magma') # Swapped to magma for better contrast
plt.axis('off')
plt.colorbar(fraction=0.046, pad=0.04)

plt.subplot(1, 2, 2)
plt.title(f"Spiking Feature Map (T=1 SFN)\nMSE: {feature_mse:.4f}")
plt.imshow(snn_map, cmap='magma')
plt.axis('off')
plt.colorbar(fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def brainstorm_channel_wise_snn(alpha=0.2, percentile=99.0):
    """
    Simulates a localized Channel-Wise Scale-and-Fire layer 
    with a residual soft-leak (alpha) to preserve the geometric core.
    """
    # 1. Fetch fresh sample data
    sample_images, _ = next(iter(val_loader))
    sample_images = sample_images.to(device)
    
    # 2. Get continuous golden baseline features
    with torch.no_grad():
        cont_features = golden_encoder(sample_images) # [B, 1280, 7, 7]
    
    # 3. Simulate localized Channel-Wise SFN Surgery
    B, C, H, W = cont_features.shape
    spiking_features_sim = cont_features.clone()
    
    for c in range(C):
        # Isolate the entire spatial distribution for just this specific channel
        channel_data = cont_features[:, c, :, :]
        
        # Calculate a localized threshold just for this channel's population
        theta_c = np.percentile(channel_data.cpu().numpy(), percentile)
        
        if theta_c > 0:
            # Step A: Generate binary spikes based on localized threshold
            spikes = (channel_data >= theta_c).float()
            scaled_spikes = spikes * theta_c
            
            # Step B: Capture sub-threshold residual voltage (the information we lost!)
            residual = channel_data * (channel_name := (channel_data < theta_c).float())
            
            # Step C: Brainstormed Hybrid Output (Spikes + Soft Residual Leak)
            # This mimics a multi-timestep accumulation where voltage isn't permanently thrown away
            hybrid_output = scaled_spikes + (alpha * residual)
            spiking_features_sim[:, c, :, :] = hybrid_output
            
    # 4. Evaluate the brainstormed fix
    sim_mse = F.mse_loss(spiking_features_sim, cont_features).item()
    
    # 5. Visualize the Total Energy
    cont_map = cont_features[0].mean(dim=0).cpu().numpy()
    sim_map = spiking_features_sim[0].mean(dim=0).cpu().numpy()
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Continuous Feature Map")
    plt.imshow(cont_map, cmap='magma')
    plt.axis('off')
    plt.colorbar(fraction=0.046, pad=0.04)
    
    plt.subplot(1, 2, 2)
    plt.title(f"Brainstormed Channel-SNN\nMSE: {sim_mse:.4f}")
    plt.imshow(sim_map, cmap='magma')
    plt.axis('off')
    plt.colorbar(fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Brainstormed Feature-Level MSE: {sim_mse:.6f}")

# Run the test loop
brainstorm_channel_wise_snn(alpha=0.1, percentile=95.0)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

def innovate_strict_t1_snn(percentile=99.0, crop_margin=2):
    """
    Simulates a strict T=1 Scale-and-Fire Network, but innovates the 
    quantization thresholds by ignoring edge-padding artifacts during profiling.
    """
    print("--- Initiating Strict T=1 Spatial-Masked Simulation ---")
    
    # 1. Fetch data and continuous features
    sample_images, _ = next(iter(val_loader))
    sample_images = sample_images.to(device)
    
    with torch.no_grad():
        cont_features = golden_encoder(sample_images) # [B, 1280, 7, 7]
    
    B, C, H, W = cont_features.shape
    strict_t1_features = torch.zeros_like(cont_features)
    
    # 2. The T=1 Innovation Loop
    for c in range(C):
        channel_data = cont_features[:, c, :, :]
        
        # --- THE HACK: Center-Cropped Profiling ---
        # We only calculate the threshold using the safe, center pixels
        # ignoring the outer 'crop_margin' where the padding artifacts live.
        if H > crop_margin*2 and W > crop_margin*2:
            safe_core = channel_data[:, crop_margin:-crop_margin, crop_margin:-crop_margin]
        else:
            safe_core = channel_data # Fallback if feature map is too tiny (e.g., 1x1)
            
        # Calculate theta using ONLY the uncorrupted core geometry
        theta_c = np.percentile(safe_core.cpu().numpy(), percentile)
        
        # --- STRICT T=1 SFN ---
        # Pure binary quantization. No soft-leaks. No multi-timesteps.
        if theta_c > 0:
            spikes = (channel_data >= theta_c).float()
            strict_t1_features[:, c, :, :] = spikes * theta_c
            
    # 3. Evaluate the new T=1 Architecture
    t1_mse = F.mse_loss(strict_t1_features, cont_features).item()
    print(f"Strict T=1 Feature-Level MSE: {t1_mse:.6f}")
    
    # 4. Visual Verification (Total Energy)
    cont_map = cont_features[0].mean(dim=0).cpu().numpy()
    t1_map = strict_t1_features[0].mean(dim=0).cpu().numpy()
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Continuous Feature Map")
    plt.imshow(cont_map, cmap='magma')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title(f"Innovated T=1 SFN\nMSE: {t1_mse:.4f}")
    plt.imshow(t1_map, cmap='magma')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Execute the simulation!
innovate_strict_t1_snn(percentile=95.0, crop_margin=2)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from tqdm.notebook import tqdm
import torchvision.models as models

print("=========================================")
print("    PHASE 4: STRICT T=1 SNN PIPELINE     ")
print("=========================================")

# 1. Procure a fresh, uncorrupted backbone and decoder
final_continuous_model = models.mobilenet_v2(weights='DEFAULT').to(device)
final_continuous_model.eval()

# 2. Define the Innovated Strict T=1 SFN Layer
class StrictT1SFN(nn.Module):
    def __init__(self, thresholds):
        super().__init__()
        # Thresholds is a 1D tensor of length C (Channels). 
        # We reshape it to [1, C, 1, 1] so PyTorch can broadcast it perfectly across the image.
        self.register_buffer('thresholds', torch.tensor(thresholds, dtype=torch.float32).view(1, -1, 1, 1))

    def forward(self, x):
        # Pure binary quantization. If voltage > channel_threshold -> Fire.
        spikes = (x >= self.thresholds).float()
        return spikes * self.thresholds

# 3. Spatial Calibration Engine
spatial_profiles = {}

def get_spatial_profile(layer_name):
    def hook(model, input, output):
        # We only need exactly one batch to calculate the spatial distribution
        if layer_name not in spatial_profiles:
            spatial_profiles[layer_name] = output.detach().cpu()
    return hook

# Step A: Safely prep the network using the Phase 1 recursive method
def make_snn_ready(module):
    for child_name, child in module.named_children():
        if isinstance(child, nn.ReLU) or isinstance(child, nn.ReLU6):
            setattr(module, child_name, nn.ReLU(inplace=False))
        else:
            make_snn_ready(child)

make_snn_ready(final_continuous_model)

# Step B: Attach probes to the newly safely-instantiated ReLUs
spatial_hooks = []
for name, module in final_continuous_model.named_modules():
    if isinstance(module, nn.ReLU):
        spatial_hooks.append(module.register_forward_hook(get_spatial_profile(name)))

# Step C: Run exactly 1 calibration batch
print("Running Spatial Calibration (1 Batch)...")
sample_images, _ = next(iter(val_loader))
with torch.no_grad():
    _ = final_continuous_model(sample_images.to(device))

# Step D: Remove probes
for handle in spatial_hooks:
    handle.remove()

# 4. The Innovated Surgery (Center-Cropped, Channel-Wise)
def apply_innovated_surgery(module, prefix="", crop_margin=2, percentile=95.0):
    for name, child in module.named_children():
        full_name = f"{prefix}.{name}" if prefix else name
        
        if isinstance(child, nn.ReLU):
            if full_name in spatial_profiles:
                # Shape: [Batch, Channels, Height, Width]
                layer_data = spatial_profiles[full_name]
                C = layer_data.shape[1]
                H = layer_data.shape[2]
                W = layer_data.shape[3]
                
                channel_thresholds = np.zeros(C)
                
                # Calculate thresholds per channel, ignoring padding artifacts
                for c in range(C):
                    channel_map = layer_data[:, c, :, :]
                    if H > crop_margin * 2 and W > crop_margin * 2:
                        safe_core = channel_map[:, crop_margin:-crop_margin, crop_margin:-crop_margin]
                    else:
                        safe_core = channel_map
                        
                    channel_thresholds[c] = np.percentile(safe_core.numpy(), percentile)
                
                # Swap the layer
                setattr(module, name, StrictT1SFN(thresholds=channel_thresholds).to(device))
        else:
            apply_innovated_surgery(child, full_name, crop_margin, percentile)

print("Executing Neuromorphic Surgery...")
apply_innovated_surgery(final_continuous_model)
print("Surgery Complete. The network is now a strictly T=1 Spiking Neural Network.")

# 5. Build the final Depth Model
final_depth_model = SimpleDepthDecoder(final_continuous_model).to(device)
final_depth_model.eval()

# 6. The Final Validation Loop
print("\n--- Running Final System Validation ---")
total_final_rmse = 0.0
num_batches = 0

with torch.no_grad():
    for images, gt_depths in tqdm(val_loader, total=MAX_BATCHES, desc="Calculating Score"):
        images = images.to(device)
        gt_depths = gt_depths.to(device)
        
        predicted_depths = final_depth_model(images)
        batch_rmse = calculate_rmse(predicted_depths, gt_depths)
        
        total_final_rmse += batch_rmse
        num_batches += 1
        
        if num_batches >= MAX_BATCHES:
            break

final_snn_rmse = total_final_rmse / num_batches

print("\n=========================================")
print("          PROPOSAL FINALIZED        ")
print("=========================================")
print(f"Continuous Golden Baseline: {golden_rmse:.4f}")
print(f"Innovated T=1 SNN RMSE:     {final_snn_rmse:.4f}")
print("=========================================")